# α-AGI Business 3 · Enterprise Studio

Turn explicit business assumptions into a constrained portfolio, stress results and reviewable job specifications. **Run all cells** on a Python 3.11–3.13 runtime. No API key, GPU, paid service or Python package installation is required for the planner.

The five examples use constructed inputs. Calculations are reproducible; they do not establish investment returns, validate the inputs or approve spending. All exported jobs are unsubmitted. Original flowcharts, manuscripts and the [research notebook archive](research_loop_archive.ipynb) remain available.

[Open the browser workspace](https://montrealai.github.io/AGI-Alpha-Agent-v0/alpha_agi_business_3_v1/) · [Read the guide](https://github.com/MontrealAI/AGI-Alpha-Agent-v0/tree/v1.13.0/alpha_factory_v1/demos/alpha_agi_business_3_v1)

## 1. Open a reproducible source checkout

The default checkout is release **v1.13.0**. It uses network access only to fetch public source. For an existing or offline checkout, set `BUSINESS3_SOURCE` to its repository root before running. An existing directory is verified and reused without overwriting it. The source commit is printed for your records.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use a Python 3.11, 3.12 or 3.13 runtime")
release = "v1.13.0"
configured_source = os.environ.get("BUSINESS3_SOURCE")
source = Path(configured_source).expanduser().resolve() if configured_source else Path.cwd() / "business3-v1.13.0"
if not source.exists():
    if configured_source:
        raise FileNotFoundError("BUSINESS3_SOURCE does not exist; choose an existing checkout")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", release,
                    "https://github.com/MontrealAI/AGI-Alpha-Agent-v0.git", str(source)], check=True)
module = source / "alpha_factory_v1/demos/alpha_agi_business_3_v1/enterprise.py"
if not module.is_file():
    raise RuntimeError("This directory does not contain the maintained Business 3 planner")
if not configured_source:
    checked_tag = subprocess.check_output(
        ["git", "-C", str(source), "describe", "--exact-match", "--tags"], text=True).strip()
    if checked_tag != release:
        raise RuntimeError("The existing source directory is not the expected release")
commit = subprocess.check_output(["git", "-C", str(source), "rev-parse", "HEAD"], text=True).strip()
print("Source commit:", commit)
sys.path.insert(0, str(source))
from alpha_factory_v1.demos.alpha_agi_business_3_v1.enterprise import read_json, solve, verify, write_bundle, brief
cases = read_json(module.with_name("scenarios.json"))
print("Cases:", ", ".join(case["id"] for case in cases))

## 2. Select a case or supply your own assumptions

Choose `industrial`, `lean-budget`, `review-bottleneck`, `severe-downside` or `evidence-first`. To use your own file, set `input_file` to its scenario JSON path. Start with a downloaded `scenario.json`; the strict schema rejects ambiguous or incomplete inputs. Cash flows and capital use whole USD; job rewards use a separate whole-AGIALPHA budget.

In [ ]:
case_name = "industrial"
input_file = None  # Example: "/content/my-scenario.json"
if input_file:
    scenario = read_json(Path(input_file))
else:
    scenario = next((case for case in cases if case["id"] == case_name), None)
    if scenario is None:
        raise ValueError("Choose a case listed above")
report = solve(scenario)
print(brief(report))

## 3. Retain and independently recompute the evidence

Each distinct run gets its own SHA-256 directory. Identical runs reuse identical files; modified or incomplete prior output is never overwritten. The seven-file bundle includes the full dossier, scenario, brief, CSV, unsubmitted jobs, an unminted and unencrypted seed draft, and checksums.

In [ ]:
output_root = Path(os.environ.get("BUSINESS3_OUTPUT", "business3-runs")).expanduser().resolve()
run_dir = write_bundle(report, output_root)
replayed = verify(read_json(run_dir / "dossier.json", 2_000_000))
assert replayed == report
print("Verified dossier:", report["sha256"])
print("Evidence directory:", run_dir)
for path in sorted(run_dir.iterdir()):
    print(" ", path.name, path.stat().st_size, "bytes")

## 4. Download and review

The ZIP contains the exact retained files. In Colab the download starts automatically. Elsewhere, use the printed local path. Independent review of source evidence and downside assumptions is still required. A valid hash proves file consistency, not real-world truth or reviewer identity.

In [ ]:
import hashlib
import zipfile

archive_path = run_dir.with_suffix(".zip")
if archive_path.exists():
    with zipfile.ZipFile(archive_path) as existing:
        expected = {path.name: path.read_bytes() for path in run_dir.iterdir()}
        if {name: existing.read(name) for name in existing.namelist()} != expected:
            raise ValueError("Existing ZIP differs; preserve it and choose another output root")
else:
    with zipfile.ZipFile(archive_path, "x", zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(run_dir.iterdir()):
            bundle.write(path, path.name)
print("ZIP:", archive_path)
print("ZIP SHA-256:", hashlib.sha256(archive_path.read_bytes()).hexdigest())
try:
    from google.colab import files
except ImportError:
    print("Open the ZIP at the path above, or import dossier.json in the browser workspace.")
else:
    files.download(str(archive_path))

## 5. Continue through the Ascension lifecycle

For a nonempty portfolio, the exported `jobs.json` contains goal ↔ success metric ↔ bounty. With the separately installed operator runtime, compile and check a FusionPlan:

```sh
alpha-agent ascension-compile path/to/jobs.json --output fusion-plan.json
alpha-agent ascension-check fusion-plan.json
```

[The protocol guide](https://montrealai.github.io/AGI-Alpha-Agent-v0/agent/ASCENSION_PROTOCOL/) covers the undeployed local-EVM reference for seed commitments, MARK risk gating, Sovereign treasury, staked ENS agent and validator roles, auctions, slashing and the 1% payout burn. This notebook does not mint, submit transactions, train model weights or confer compliance approval.

**Recovery:** fix an invalid input and rerun from step 2. If an old run or ZIP was altered, retain it and choose a new `BUSINESS3_OUTPUT`. If source fetching fails, set `BUSINESS3_SOURCE` to a previously downloaded checkout and rerun step 1. [Read the complete guide](https://github.com/MontrealAI/AGI-Alpha-Agent-v0/tree/v1.13.0/alpha_factory_v1/demos/alpha_agi_business_3_v1).